# Chapter 9: Text Processing and Multiclass Classification
## 📌 Summary
Text processing mengubah raw text menjadi representasi numerik yang bisa digunakan model ML. Teknik utama: **CountVectorizer**, **TF-IDF**, dan **text pipelines**.

## 🧠 Theoretical Deep-Dive

### 9.1 Bag of Words (BoW)
Representasikan dokumen sebagai vektor frekuensi kata. Mengabaikan urutan kata.
`CountVectorizer`: menghasilkan raw count matrix.

### 9.2 TF-IDF
TF-IDF = TF × IDF
- **TF** (Term Frequency): frekuensi kata dalam dokumen
- **IDF** (Inverse Document Frequency): log(N/df) → kata yang jarang = lebih informatif

TF-IDF menurunkan bobot kata umum (the, is, a) dan menaikkan kata yang distinctive.

### 9.3 N-grams
Extend BoW dengan urutan kata: unigram (1 kata), bigram (2 kata), trigram (3 kata). Menangkap konteks lokal.

### 9.4 Multiclass Strategies
- **OneVsRest**: k classifiers, satu per kelas
- **OneVsOne**: k(k-1)/2 classifiers, semua pasangan kelas
- **Multinomial NB**: native multiclass, dirancang untuk text

### 9.5 Naive Bayes untuk Text
P(class|text) ∝ P(class) × Π P(word|class)
Asumsi: kata independent given class (Naive). Tapi sangat efektif untuk text classification.

## 💻 Code Reproduction

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "machine learning is fun",
    "learning python for machine learning",
    "python is great for data science",
    "deep learning and machine learning"
]

cv = CountVectorizer()
X = cv.fit_transform(corpus)

print("Vocabulary:", cv.vocabulary_)
print("Feature names:", cv.get_feature_names_out())
print("Document-term matrix (dense):")
print(X.toarray())
print("Shape:", X.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = [
    "machine learning is fun",
    "learning python for machine learning",
    "python is great for data science",
    "deep learning and machine learning"
]

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print("Feature names:", tfidf.get_feature_names_out())
print("\nTF-IDF matrix:")
print(X_tfidf.toarray().round(3))
print("\nHighest TF-IDF words per doc:")
for i, doc in enumerate(corpus):
    scores = X_tfidf[i].toarray()[0]
    top_idx = np.argsort(scores)[::-1][:3]
    top_words = [(tfidf.get_feature_names_out()[j], scores[j].round(3)) for j in top_idx if scores[j] > 0]
    print(f"Doc {i}: {top_words}")

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

categories = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.misc"]
train = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

print("Train samples:", len(train.data))
print("Test samples:", len(test.data))
print("Categories:", train.target_names)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words="english")),
    ("clf", MultinomialNB(alpha=0.1))
])

pipe.fit(train.data, train.target)
pred = pipe.predict(test.data)
print("\nAccuracy:", (pred == test.target).mean().round(4))
print("\nClassification Report:")
print(classification_report(test.target, pred, target_names=categories))

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

categories = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.misc"]
train = fetch_20newsgroups(subset="train", categories=categories, remove=("headers","footers","quotes"))
test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers","footers","quotes"))

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words="english")

classifiers = [
    ("Naive Bayes", MultinomialNB(alpha=0.1)),
    ("Logistic Regression", LogisticRegression(max_iter=1000, C=10)),
    ("Linear SVC", LinearSVC(C=1.0, max_iter=2000))
]

for name, clf in classifiers:
    pipe = Pipeline([("tfidf", tfidf), ("clf", clf)])
    pipe.fit(train.data, train.target)
    acc = pipe.score(test.data, test.target)
    print(f"{name:25}: accuracy={acc:.4f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = [
    "machine learning is exciting and fun",
    "python is great for machine learning algorithms",
    "deep learning neural networks are powerful"
]

# N-gram comparison
for ngram_range in [(1,1), (1,2), (2,2)]:
    tfidf = TfidfVectorizer(ngram_range=ngram_range, max_features=20)
    X = tfidf.fit_transform(corpus)
    print(f"n-gram={ngram_range}: {len(tfidf.get_feature_names_out())} features")
    print("Features:", tfidf.get_feature_names_out()[:10], "...\n")

## ✅ Chapter Summary
- **CountVectorizer** → raw counts; **TF-IDF** → penalized by document frequency
- **N-grams** menangkap konteks lokal (bigrams sering meningkatkan performa)
- **Multinomial Naive Bayes** → sangat efisien dan efektif untuk text (baseline kuat)
- **Linear SVC** + TF-IDF → sering best untuk text classification
- Gunakan `stop_words='english'` dan `max_features` untuk efisiensi
- Pipeline text+classifier → mudah deploy dan evaluate